# Xây dựng và đánh giá hệ thống phát hiện lời phàn nàn trong đánh giá sản phẩm tiếng Việt

Notebook chạy lại toàn bộ quy trình trên Kaggle: baselines TF-IDF, kiểm tra độ bền, và fine-tune PhoBERT bằng GPU với ba seed được cố định trước (42, 43, 44). Đơn vị dự đoán là toàn bộ đánh giá (có/không có phàn nàn); đây không phải bài toán gán nhãn từng đoạn.

Dữ liệu UIT-ViOCD được tải theo commit cố định. Phân chia gốc được giữ nguyên; lọc trùng và xử lý chồng lấn chỉ dùng nội dung văn bản. Validation chọn epoch; test chỉ dùng báo cáo cuối. Test này đã được xem ở vòng thực nghiệm trước, vì vậy lần chạy ba seed là phép lặp có kiểm soát trên cùng test, không phải xác nhận độc lập trên dữ liệu mới.

Chọn GPU trong phần Notebook options/Settings của Kaggle, bật Internet để tải bộ dữ liệu và trọng số PhoBERT, rồi chạy tất cả. Notebook sẽ dừng rõ ràng nếu CUDA không khả dụng. Kết quả, log, dự đoán theo từng dòng và trọng số các tầng được fine-tune được lưu trong /kaggle/working để tải về.


In [ ]:
%pip install -q pandas==2.3.3 scikit-learn==1.8.0 underthesea==9.5.0 matplotlib==3.10.8 transformers==4.51.3


## Quy trình và đánh giá

Giữ tập chia gốc, lọc dữ liệu trùng, học TF-IDF trên train, chọn cấu hình trên validation rồi so sánh trên test. Các bảng kết quả được lưu trong thư mục `results`. [Bài báo](https://arxiv.org/abs/2104.11969) · [Dữ liệu](https://huggingface.co/datasets/tarudesu/ViOCD).

In [ ]:
"""Run CPU baselines and train/evaluate PhoBERT on CUDA for UIT-ViOCD."""

import unicodedata
import json
import numpy as np
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from underthesea import word_tokenize

SOURCE = "https://huggingface.co/datasets/tarudesu/ViOCD/resolve/25e1851683ebc407d28e5a2d514b7ae0eadd4e4b/"
LABELS = [0, 1]
NAMES = ["khong phan nan", "co phan nan"]


def normalize(text):
    return " ".join(unicodedata.normalize("NFC", text).lower().split())


def load_splits():
    splits = {}
    seen = set()
    audit = {}
    # Preserve test first, then validation. Remove overlaps from training.
    for name in ("test", "val", "train"):
        with urlopen(SOURCE + name + ".csv", timeout=30) as response:
            raw = pd.read_csv(response)
        data = raw.dropna(subset=["review", "label"]).copy()
        missing = len(raw) - len(data)
        data["label"] = data["label"].astype(int)
        data["text"] = data["review"].map(normalize)
        blank = int(data["text"].eq("").sum())
        data = data.loc[data["text"].ne("")]
        conflicts = data.groupby("text")["label"].transform("nunique").gt(1)
        consistent = data.loc[~conflicts]
        data = consistent.drop_duplicates("text")
        overlap = data["text"].isin(seen)
        data = data.loc[~overlap].copy()
        seen.update(data["text"])
        splits[name] = data
        lengths = data["text"].str.split().str.len()
        audit[name] = {
            "raw": len(raw), "missing": missing, "blank": blank,
            "conflict_rows": int(conflicts.sum()),
            "duplicate_rows": len(consistent) - (len(data) + int(overlap.sum())),
            "overlap_rows": int(overlap.sum()), "retained": len(data),
            "labels": data["label"].value_counts().sort_index().to_dict(),
            "domains": data["domain"].value_counts().to_dict(),
            "whitespace_units": int(lengths.sum()),
            "median_length": float(lengths.median()),
            "p95_length": float(lengths.quantile(.95)),
        }
        print(name, "raw:", len(raw), "retained:", len(data),
              "labels:", data["label"].value_counts().sort_index().to_dict(), flush=True)
    return splits, audit


def audit_segmented_split_overlaps(splits):
    """Fail if word segmentation creates duplicate inputs within/across splits."""
    seen_labels = {}
    segmented_splits = {}
    rows = []
    for name in ("test", "val", "train"):
        frame = splits[name]
        segmented = frame["text"].map(lambda text: word_tokenize(text, format="text"))
        segmented_splits[name] = segmented
        duplicate_rows = int(segmented.duplicated(keep="first").sum())
        cross_overlap = segmented.isin(seen_labels)
        local = pd.DataFrame({"segmented": segmented, "label": frame["label"].to_numpy()})
        conflicting = set(local.groupby("segmented")["label"].nunique().loc[lambda counts: counts.gt(1)].index)
        within_conflict_rows = int(segmented.isin(conflicting).sum())
        cross_conflict_rows = int(sum(
            bool(overlap) and seen_labels.get(segment) != int(label)
            for segment, label, overlap in zip(segmented, frame["label"], cross_overlap)
        ))
        result = {
            "split": name, "rows": int(len(frame)),
            "within_split_duplicate_segmented_rows": duplicate_rows,
            "cross_split_segmented_overlap_rows": int(cross_overlap.sum()),
            "within_split_segment_label_conflict_rows": within_conflict_rows,
            "cross_split_segment_label_conflict_rows": cross_conflict_rows,
        }
        rows.append(result)
        if any(result[key] for key in result if key.endswith("rows") and key != "rows"):
            raise ValueError(f"Segmented model-input overlap/conflict detected: {result}")
        seen_labels.update({segment: int(label) for segment, label in zip(segmented, frame["label"])})
    return rows, segmented_splits


def select_error_examples(errors, per_label=5, random_seed=42):
    """Sample domain-stratified errors proportionally within each true label."""
    selected = []
    for label, label_errors in errors.groupby("label", sort=True):
        counts = label_errors["domain"].value_counts().sort_index()
        target = min(per_label, len(label_errors))
        exact = counts * target / len(label_errors)
        quotas = exact.apply(np.floor).astype(int)
        remaining = target - int(quotas.sum())
        order = sorted(counts.index, key=lambda domain: (-(exact[domain] - quotas[domain]), str(domain)))
        for domain in order:
            if remaining == 0:
                break
            if quotas[domain] < counts[domain]:
                quotas[domain] += 1
                remaining -= 1
        for domain_index, domain in enumerate(sorted(counts.index, key=str)):
            quota = int(quotas[domain])
            if quota:
                stratum = label_errors.loc[label_errors["domain"].eq(domain)]
                selected.append(stratum.sample(
                    n=quota, random_state=random_seed + int(label) * 1000 + domain_index
                ))
    if not selected:
        return errors.head(0).copy()
    return pd.concat(selected).sort_values(["label", "domain"], kind="stable")


def main(results_dir="results"):
    splits, audit = load_splits()
    segmented_overlap_audit, segmented = audit_segmented_split_overlaps(splits)
    train, val, test = (splits[name] for name in ("train", "val", "test"))
    # Use the same features for the model comparison and the same SVM for
    # the segmentation comparison. Fit TF-IDF exclusively on training text.
    configurations = {
        "A_LogReg_space": (LogisticRegression(max_iter=1000), False),
        "B_SVM_space": (LinearSVC(random_state=42), False),
        "C_SVM_segmented": (LinearSVC(random_state=42), True),
    }
    models, val_scores = {}, {}
    for name, (classifier, use_segmentation) in configurations.items():
        model = make_pipeline(
            TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                            ngram_range=(1, 2), min_df=2),
            classifier,
        )
        model.fit(segmented["train"] if use_segmentation else train["text"], train["label"])
        predictions = model.predict(segmented["val"] if use_segmentation else val["text"])
        val_scores[name] = f1_score(val["label"], predictions, average="macro")
        models[name] = model
    best = max(val_scores, key=val_scores.get)
    print("Selected on validation:", best)
    rows, test_predictions = [], {}
    for name, model in models.items():
        predictions = model.predict(segmented["test"] if configurations[name][1] else test["text"])
        test_predictions[name] = predictions
        metrics = classification_report(test["label"], predictions, labels=LABELS,
                                        target_names=NAMES, output_dict=True, zero_division=0)
        rows.append({
            "configuration": name,
            "val_macro_f1": val_scores[name],
            "test_macro_f1": metrics["macro avg"]["f1-score"],
            "test_accuracy": metrics["accuracy"],
            "complaint_precision": metrics[NAMES[1]]["precision"],
            "complaint_recall": metrics[NAMES[1]]["recall"],
            "complaint_f1": metrics[NAMES[1]]["f1-score"],
        })
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    predictions = test_predictions[best]
    matrix = confusion_matrix(test["label"], predictions, labels=LABELS)
    print("Confusion matrix [0, 1]:\n", matrix)
    errors = test.assign(predicted=predictions)
    errors = errors.loc[errors["label"].ne(errors["predicted"])]
    domain_order = sorted(test["domain"].dropna().unique().tolist())
    error_index = pd.MultiIndex.from_product([LABELS, domain_order], names=["label", "domain"])
    error_counts = errors.groupby(["label", "domain"]).size().reindex(
        error_index, fill_value=0
    ).rename("error_count").reset_index()
    print("\nAll baseline test errors by true label and domain:")
    print(error_counts.to_string(index=False))
    print("\nIllustrative error examples (up to five per class; proportional domain strata; seed 42):")
    examples = select_error_examples(errors, per_label=5, random_seed=42)[
        ["label", "predicted", "domain", "text"]
    ].copy()
    examples.insert(0, "test_row", examples.index.astype(int))
    examples["text_excerpt"] = examples["text"].fillna("").astype(str).str.slice(0, 180)
    examples = examples[["test_row", "label", "predicted", "domain", "text_excerpt"]]
    print(examples.to_string(index=False))
    destination = Path(results_dir)
    destination.mkdir(parents=True, exist_ok=True)
    error_counts.to_csv(destination / "error_counts_by_domain.csv", index=False)
    (destination / "segmented_overlap_audit.json").write_text(
        json.dumps(segmented_overlap_audit, indent=2), encoding="utf-8")
    error_summary = {
        str(label): {
            str(row.domain): int(row.error_count)
            for row in error_counts.itertuples(index=False)
            if int(row.label) == int(label)
        }
        for label in LABELS
    }
    (destination / "summary.json").write_text(
        json.dumps({
            "source": SOURCE, "data": audit, "segmented_overlap_audit": segmented_overlap_audit, "models": rows,
            "selected": best, "confusion_matrix": matrix.tolist(),
            "error_counts_by_true_label_and_domain": error_summary,
            "error_example_selection": {
                "method": "largest-remainder proportional allocation by domain within each true-label error class",
                "max_examples_per_class": 5, "random_seed": 42,
                "selected_test_row_ids": examples["test_row"].astype(int).tolist(),
                "interpretation": "Illustrative examples only; improved domain coverage does not make this a representative sample.",
            },
        }, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    examples.drop(columns="text_excerpt").set_index("test_row").to_csv(
        destination / "error_ids.csv", index_label="test_row"
    )
    examples.to_csv(destination / "error_examples.csv", index=False)
    pd.DataFrame(rows).to_csv(destination / "metrics.csv", index=False)
    pd.DataFrame({"test_row": test.index, "label": test["label"], "domain": test["domain"],
                  "prediction_baseline": predictions}).to_csv(destination / "baseline_test_predictions.csv", index=False)
    all_predictions = pd.DataFrame({
        "test_row": test.index, "label": test["label"], "domain": test["domain"],
    })
    for name, model_predictions in test_predictions.items():
        all_predictions[f"prediction_{name}"] = model_predictions
    all_predictions.to_csv(destination / "baseline_all_test_predictions.csv", index=False)


In [ ]:
main()

## Khảo sát quy mô tập huấn luyện

Cấu hình C được huấn luyện trên 25%, 50%, 75% và 100% tập train. Ba mức dưới 100% dùng lấy mẫu phân tầng theo nhãn với năm seed (42–46); validation được giữ cố định và dùng cùng một cấu hình TF-IDF/Linear SVM. Chỉ Macro-F1 validation được báo cáo; test không tham gia lựa chọn hay phép đo này. SD ở các mức lặp mô tả biến thiên giữa các tập con lấy mẫu; mức 100% chỉ có một lượt nên không có SD.


In [ ]:
from sklearn.model_selection import train_test_split


def run_learning_curve(results_dir="results"):
    splits, _ = load_splits()
    train, val = splits["train"], splits["val"]
    train_features = train["text"].map(lambda text: word_tokenize(text, format="text"))
    val_features = val["text"].map(lambda text: word_tokenize(text, format="text"))
    raw_rows, manifest_rows = [], []
    plan = ((0.25, 5), (0.50, 5), (0.75, 5), (1.00, 1))

    for fraction, repetitions in plan:
        n_train = int(len(train) * fraction)
        for repeat in range(repetitions):
            seed = 42 + repeat
            if fraction == 1.0:
                row_ids = train.index.tolist()
            else:
                row_ids, _ = train_test_split(
                    train.index, train_size=n_train, random_state=seed,
                    stratify=train["label"],
                )
                row_ids = sorted(row_ids.tolist())
            model = make_pipeline(
                TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                                ngram_range=(1, 2), min_df=2),
                LinearSVC(random_state=42),
            )
            model.fit(train_features.loc[row_ids], train.loc[row_ids, "label"])
            prediction = model.predict(val_features)
            score = f1_score(val["label"], prediction, labels=LABELS,
                             average="macro", zero_division=0)
            raw_rows.append({
                "train_fraction": fraction, "train_size": n_train,
                "repeat": repeat + 1, "random_seed": seed,
                "val_macro_f1": float(score),
            })
            manifest_rows.extend({
                "train_fraction": fraction, "repeat": repeat + 1,
                "random_seed": seed, "train_row": row_id,
            } for row_id in row_ids)

    raw = pd.DataFrame(raw_rows)
    summary = raw.groupby(["train_fraction", "train_size"], as_index=False).agg(
        repetitions=("val_macro_f1", "size"),
        val_macro_f1_mean=("val_macro_f1", "mean"),
        val_macro_f1_sd_sample=("val_macro_f1", "std"),
    )
    destination = Path(results_dir)
    destination.mkdir(parents=True, exist_ok=True)
    raw.to_csv(destination / "learning_curve_runs.csv", index=False)
    summary.to_csv(destination / "learning_curve_summary.csv", index=False)
    pd.DataFrame(manifest_rows).to_csv(destination / "learning_curve_samples.csv", index=False)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4.2), layout="constrained")
    x = 100 * summary["train_fraction"].to_numpy()
    y = 100 * summary["val_macro_f1_mean"].to_numpy()
    sd = 100 * summary["val_macro_f1_sd_sample"].fillna(0).to_numpy()
    ax.errorbar(x, y, yerr=sd, marker="o", capsize=4, color="#176b87")
    tick_labels = [f"{fraction:.0f}%\nn={size:,}, k={reps}"
                   for fraction, size, reps in zip(x, summary["train_size"], summary["repetitions"])]
    ax.set(xlabel="Tỷ lệ tập train (n = số mẫu; k = số lượt)", ylabel="Macro-F1 validation (%)",
           title="Khảo sát quy mô tập huấn luyện", xticks=x, xticklabels=tick_labels,
           xlim=(18, 110), ylim=(87, 93.5))
    ax.tick_params(axis="x", labelsize=8)
    ax.grid(axis="y", alpha=.25)
    for xi, yi, sigma in zip(x, y, sd):
        ax.annotate(f"{yi:.2f}%", (xi, yi + sigma + 0.12), ha="center", va="bottom",
                    fontsize=8, bbox={"facecolor":"white", "edgecolor":"none", "alpha":.85, "pad":1})
    fig.savefig(destination / "learning_curve.png", dpi=180)
    plt.show()
    print(summary.round(4).to_string(index=False))
    return summary


learning_curve_summary = run_learning_curve()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

summary = json.loads(Path("results/summary.json").read_text(encoding="utf-8"))
scores = pd.DataFrame(summary["models"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
positions = range(len(scores))
axes[0].bar([x - .18 for x in positions], 100 * scores["val_macro_f1"], .36, label="Validation")
axes[0].bar([x + .18 for x in positions], 100 * scores["test_macro_f1"], .36, label="Test")
axes[0].set(xticks=list(positions), xticklabels=["A", "B", "C"], ylabel="Macro-F1 (%)",
            ylim=(0, 100), title="So sánh ba cấu hình")
axes[0].legend()
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay(
    confusion_matrix=np.asarray(summary["confusion_matrix"]),
    display_labels=["Không phàn nàn", "Có phàn nàn"],
).plot(ax=axes[1], cmap="Blues", colorbar=False)
axes[1].set(title="Ma trận nhầm lẫn trên test", xlabel="Dự đoán", ylabel="Nhãn thật")
fig.savefig("results/evaluation.png", dpi=170)
with plt.rc_context({"svg.fonttype": "none"}):
    fig.savefig("results/evaluation.svg")
plt.show()

## Kiểm tra độ bền và giới hạn bản bỏ dấu

So sánh C và E trên bản gốc, bản bỏ dấu và 18 cặp câu chẩn đoán; kiểm tra miền giữ lại từng ngành và đối chứng cùng số mẫu từng lớp qua ba seed. Bản bỏ dấu có thể làm trùng văn bản giữa train và test: notebook kiểm toán các ID bị va chạm, báo sensitivity loại các ID này, và giữ kết quả 546 mẫu chỉ để đối chiếu với thực nghiệm trước. Thực nghiệm PhoBERT chính dùng nguyên văn có dấu theo split gốc; các phép thử bỏ dấu là phân tích độ bền, không phải kết quả test độc lập.


In [ ]:
"""Evaluate robustness under accent removal and held-out domains."""
import json
import unicodedata
from pathlib import Path
from statistics import mean

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.svm import LinearSVC



def remove_accents(text):
    decomposed = unicodedata.normalize("NFD", text).replace("đ", "d").replace("Đ", "D")
    return "".join(char for char in decomposed if not unicodedata.combining(char))


PAIRS = [
    ("phu_dinh", "Máy bị lỗi khi sạc.", "Máy không bị lỗi khi sạc."),
    ("phu_dinh", "Kem làm da bị rát.", "Kem không làm da bị rát."),
    ("phu_dinh", "Ứng dụng hay bị thoát giữa chừng.", "Ứng dụng chưa bao giờ bị thoát giữa chừng."),
    ("phu_dinh", "Áo bị phai màu sau khi giặt.", "Áo không bị phai màu sau khi giặt."),
    ("phu_dinh", "Điện thoại bị nóng khi gọi video.", "Điện thoại không bị nóng khi gọi video."),
    ("phu_dinh", "Giao hàng bị trễ hẹn.", "Giao hàng không bị trễ hẹn."),
    ("tuong_phan", "Áo đẹp nhưng đường may bị bung.", "Áo đẹp và đường may chắc."),
    ("tuong_phan", "Giao nhanh nhưng shop gửi sai màu.", "Giao nhanh và shop gửi đúng màu."),
    ("tuong_phan", "Giao diện đẹp nhưng ứng dụng không lưu được ảnh.", "Giao diện đẹp và ứng dụng lưu được ảnh."),
    ("tuong_phan", "Điện thoại đẹp nhưng loa rè.", "Điện thoại đẹp và loa rõ."),
    ("tuong_phan", "Mùi dễ chịu nhưng dùng xong bị nổi mẩn.", "Mùi dễ chịu và dùng xong không bị nổi mẩn."),
    ("tuong_phan", "Đóng gói kỹ nhưng thiếu dây sạc.", "Đóng gói kỹ và có đủ dây sạc."),
    ("sai_lech", "Tôi đặt hai áo, hộp chỉ có một áo.", "Tôi đặt hai áo, hộp có đủ hai áo."),
    ("sai_lech", "Đặt màu đen, nhận màu trắng.", "Đặt màu đen, nhận đúng màu đen."),
    ("sai_lech", "Hẹn thứ hai, tới thứ năm hàng mới đến.", "Hẹn thứ hai và hàng đến đúng thứ hai."),
    ("sai_lech", "Chai ghi 100 ml mà mở ra chỉ còn nửa chai.", "Chai ghi 100 ml, nhận nguyên chai đầy."),
    ("sai_lech", "Tôi chọn cỡ M, shop gửi cỡ S.", "Tôi chọn cỡ M, shop gửi đúng cỡ M."),
    ("sai_lech", "Thanh toán gói một năm nhưng tài khoản chỉ được một tháng.", "Thanh toán gói một năm và tài khoản được đủ một năm."),
]


def score(y, pred):
    return float(f1_score(y, pred, labels=[0, 1], average="macro", zero_division=0))


def pair_texts():
    return [normalize(sentence) for _, positive, negative in PAIRS for sentence in (positive, negative)]


def evaluate_pairs(pred):
    rows = []
    for i, (group, complaint, ordinary) in enumerate(PAIRS):
        a, b = map(int, pred[2*i:2*i+2])
        rows.append(dict(id=i+1, group=group, complaint=complaint, ordinary=ordinary,
                         complaint_prediction=a, ordinary_prediction=b, both_correct=(a,b)==(1,0)))
    return rows


def build_model(use_chars):
    word = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False,
                           ngram_range=(1, 2), min_df=2)
    if use_chars:
        chars = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2, lowercase=False)
        features = FeatureUnion([("word", word), ("char", chars)])
    else:
        features = word
    return make_pipeline(features, LinearSVC(random_state=42))


def segmented(items):
    return [word_tokenize(text, format="text") for text in items]


def run(results_dir):
    splits, _ = load_splits()
    train, val, test = (splits[k] for k in ("train","val","test"))
    split_texts = {k: segmented(frame["text"]) for k,frame in splits.items()}
    noisy = segmented(test["text"].map(remove_accents))
    accent_overlap_diagnostics = {}
    for stage in ("after_accent_removal", "after_segmentation"):
        views = {name: frame["text"].map(remove_accents).tolist()
                 for name, frame in splits.items()}
        if stage == "after_segmentation":
            views = {name: segmented(texts) for name, texts in views.items()}
        labels_by_text = {}
        for text, label in zip(views["train"], splits["train"]["label"]):
            labels_by_text.setdefault(text, set()).add(int(label))
        by_split = {}
        for split_name in ("val", "test"):
            hits = [
                (int(index), int(label), labels_by_text[text])
                for index, text, label in zip(
                    splits[split_name].index, views[split_name], splits[split_name]["label"])
                if text in labels_by_text
            ]
            by_split[split_name] = {
                "overlap_ids": [index for index, _, _ in hits],
                "disagreeing_label_ids": [
                    index for index, label, train_labels in hits if label not in train_labels
                ],
            }
        accent_overlap_diagnostics[stage] = {
            **by_split,
            "conflicting_train_groups": sum(
                len(labels) > 1 for labels in labels_by_text.values())
        }
    overlap_ids = accent_overlap_diagnostics["after_segmentation"]["test"]["overlap_ids"]
    overlap_mask = test.index.isin(overlap_ids)
    sensitivity = test.loc[~overlap_mask]
    noisy_sensitivity = segmented(sensitivity["text"].map(remove_accents))
    accent_overlap_audit = {
        "diagnostics": accent_overlap_diagnostics,
        "sensitivity_test_rows": int(len(sensitivity)),
        "rule": "drop test rows whose accent-stripped, Underthesea-segmented text occurs in train",
    }
    challenge = segmented(pair_texts())
    rows, domains, pairs, sample_ids = [], [], [], []
    for name, use_chars in (("C_word", False), ("E_word_char", True)):
        model = build_model(use_chars).fit(split_texts["train"], train["label"])
        pair_rows = evaluate_pairs(model.predict(challenge))
        for row in pair_rows:
            pairs.append({"model":name, **row})
        rows.append({
            "model": name, "val": score(val["label"], model.predict(split_texts["val"])),
            "test_clean": score(test["label"], model.predict(split_texts["test"])),
            "test_unaccented": score(test["label"], model.predict(noisy)),
            "test_unaccented_sensitivity": score(sensitivity["label"], model.predict(noisy_sensitivity)),
            "sensitivity_test_n": int(len(sensitivity)),
            "accent_overlap_test_n": int(overlap_mask.sum()),
            "pairs_correct": sum(row["both_correct"] for row in pair_rows), "pairs_total":len(PAIRS),
        })
        for domain in sorted(test["domain"].unique()):
            training = train["domain"].ne(domain).to_numpy()
            testing = test["domain"].eq(domain).to_numpy()
            X = [text for text, keep in zip(split_texts["test"], testing) if keep]
            restricted = build_model(use_chars).fit(
                [text for text, keep in zip(split_texts["train"],training) if keep],
                train.loc[training,"label"],
            )
            same_size = []
            same_size_runs = []
            counts = train.loc[training, "label"].value_counts().to_dict()
            for seed in (42, 43, 44):
                rng = np.random.default_rng(seed)
                ids = np.concatenate([rng.choice(np.flatnonzero(train["label"].eq(label)),
                                                 count, replace=False)
                                      for label, count in sorted(counts.items())])
                matched = build_model(use_chars).fit(
                    [split_texts["train"][i] for i in ids], train["label"].iloc[ids]
                )
                value = score(test.loc[testing,"label"], matched.predict(X))
                same_size.append(value)
                sampled = train.index.to_numpy()[ids].astype(int).tolist()
                sample_ids.append({
                    "model": name, "domain": domain, "seed": seed, "macro_f1": value,
                    "train_source_rows_json": json.dumps(sampled, separators=(",", ":")),
                })
                same_size_runs.append({
                    "seed": seed, "macro_f1": value, "sample_n": len(ids),
                    "train_source_rows_file": "same_size_samples.csv",
                })
            domains.append({
                "model":name, "domain":domain, "train_heldout_n":int(training.sum()),
                "test_n":int(testing.sum()), "test_complaints":int(test.loc[testing,"label"].sum()),
                "all_domains":score(test.loc[testing,"label"], model.predict(X)),
                "heldout_domain":score(test.loc[testing,"label"], restricted.predict(X)),
                "same_size_mixed":mean(same_size),
                "same_size_mixed_sd":float(np.std(same_size, ddof=1)),
                "same_size_mixed_runs":same_size_runs,
            })
    results = Path(results_dir)
    results.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(pairs).to_csv(results / "challenge_predictions.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(sample_ids).to_csv(
        results / "same_size_samples.csv", index=False, sep=";", decimal=",", encoding="utf-8-sig")
    payload = {
        "scores": rows, "domains": domains,
        "same_size_sample_manifest": "same_size_samples.csv",
        "accent_overlap_audit": accent_overlap_audit,
    }
    (results / "robustness.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(pd.DataFrame(rows).round(4).to_string(index=False),flush=True)
    print(pd.DataFrame(domains).round(4).to_string(index=False),flush=True)

In [ ]:
run("results_robustness")

## Thực nghiệm D · PhoBERT fine-tune trên GPU Kaggle

Giao thức được cố định trước lần chạy này: cùng split và tokenizer; seed 42, 43, 44; mở khóa hai encoder block cuối và đầu phân loại; tối đa ba epoch; chọn epoch bằng Macro-F1 validation, nếu hòa chọn epoch sớm hơn. Gradient của mỗi bước tích lũy được chuẩn hóa theo đúng số mẫu trong nhóm micro-batch, kể cả nhóm cuối 8+8+8+5. Chỉ sau khi chọn checkpoint mới đánh giá test. Không dừng sớm theo test.


In [ ]:
"""Fine-tune PhoBERT on Kaggle CUDA with sample-weighted gradient accumulation."""
import json
import math
import time
from pathlib import Path

import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup, set_seed

MODEL = "vinai/phobert-base"
REVISION = "01daacda68afe13d83023d16ec647239e344a1e6"
SEEDS = (42, 43, 44)
EPOCHS = 3
MAX_LENGTH = 128
MICRO_BATCH = 8
ACCUMULATION = 4
LEARNING_RATE = 3e-5


def cuda_batch(batch):
    return {key: value.to("cuda", non_blocking=True) for key, value in batch.items()}


@torch.inference_mode()
def predict(model, loader):
    model.eval()
    predictions = []
    for batch in loader:
        batch.pop("labels", None)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**cuda_batch(batch)).logits
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return predictions


def fine_tune(seed, results_dir="results_phobert"):
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required. Enable a GPU accelerator in Kaggle Notebook options.")
    set_seed(seed)
    torch.set_num_threads(4)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.cuda.reset_peak_memory_stats()
    print("CUDA device:", torch.cuda.get_device_name(0),
          "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda,
          "| seed:", seed, flush=True)
    start = time.perf_counter()
    folder = Path(results_dir)
    folder.mkdir(parents=True, exist_ok=True)
    splits, audit = load_splits()
    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, revision=REVISION, num_labels=2)
    model.requires_grad_(False)
    model.roberta.encoder.layer[-2:].requires_grad_(True)
    model.classifier.requires_grad_(True)
    model.to("cuda")
    trainable = {name: p for name, p in model.named_parameters() if p.requires_grad}
    count = sum(p.numel() for p in trainable.values())
    print("Trainable parameters:", count, "/", sum(p.numel() for p in model.parameters()), flush=True)

    sequence_length_rows = []

    def make_loader(data, split_name, shuffle=False):
        texts = data["text"].map(lambda value: word_tokenize(value, format="text")).tolist()
        if seed == SEEDS[0]:
            full_ids = tokenizer(texts, truncation=False)["input_ids"]
            lengths = pd.Series([len(input_ids) for input_ids in full_ids], dtype="int64")
            truncated = int(lengths.gt(MAX_LENGTH).sum())
            sequence_length_rows.append({
                "split": split_name, "n_examples": int(len(lengths)),
                "includes_special_tokens": True, "max_input_tokens": MAX_LENGTH,
                "examples_truncated": truncated,
                "truncated_percent": float(100 * truncated / len(lengths)),
                "median_tokens": float(lengths.median()),
                "p95_tokens": float(lengths.quantile(.95)),
                "max_tokens_before_truncation": int(lengths.max()),
            })
        encodings = tokenizer(texts, max_length=MAX_LENGTH, truncation=True)
        items = [{**{key: values[i] for key, values in encodings.items()}, "labels": int(label)}
                 for i, label in enumerate(data["label"])]
        return DataLoader(items, batch_size=MICRO_BATCH, shuffle=shuffle, num_workers=0,
                          pin_memory=True, collate_fn=lambda rows: tokenizer.pad(
                              rows, padding=True, pad_to_multiple_of=8, return_tensors="pt"))

    train = make_loader(splits["train"], "train", shuffle=True)
    val = make_loader(splits["val"], "val")
    steps_per_epoch = math.ceil(len(train) / ACCUMULATION)
    total_steps = steps_per_epoch * EPOCHS
    optimizer = AdamW(trainable.values(), lr=LEARNING_RATE, weight_decay=.01, fused=True)
    schedule = get_linear_schedule_with_warmup(optimizer, int(.1 * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda")
    history, best_weights, best_score, best_epoch = [], None, -1.0, None
    for epoch in range(1, EPOCHS + 1):
        tick = time.perf_counter()
        model.train()
        model.roberta.embeddings.eval()
        for block in model.roberta.encoder.layer[:-2]:
            block.eval()
        optimizer.zero_grad(set_to_none=True)
        weighted_loss = 0.0
        for step, batch in enumerate(train, 1):
            group_start = ((step - 1) // ACCUMULATION) * ACCUMULATION
            group_samples = min(ACCUMULATION * MICRO_BATCH,
                                len(splits["train"]) - group_start * MICRO_BATCH)
            batch_samples = len(batch["labels"])
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**cuda_batch(batch)).loss
            scaler.scale(loss * batch_samples / group_samples).backward()
            weighted_loss += loss.detach().item() * batch_samples
            if step % ACCUMULATION == 0 or step == len(train):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable.values(), 1.0)
                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= scale_before:
                    schedule.step()
                optimizer.zero_grad(set_to_none=True)
            if step % 100 == 0:
                print(f"Epoch {epoch}/{EPOCHS} | batch {step}/{len(train)} | loss {loss.item():.4f}", flush=True)
        predictions = predict(model, val)
        val_macro = float(f1_score(splits["val"]["label"], predictions, average="macro"))
        row = {"epoch": epoch, "train_loss": weighted_loss / len(splits["train"]),
               "val_macro_f1": val_macro, "epoch_seconds": round(time.perf_counter() - tick, 2)}
        history.append(row)
        print(json.dumps(row), flush=True)
        if val_macro > best_score:
            best_score, best_epoch = val_macro, epoch
            best_weights = {name: value.detach().cpu().clone() for name, value in trainable.items()}

    for name, value in best_weights.items():
        trainable[name].data.copy_(value)
    test = splits["test"]
    predicted = predict(model, make_loader(test, "test"))
    if seed == SEEDS[0]:
        pd.DataFrame(sequence_length_rows).to_csv(
            Path(results_dir).parent / "sequence_length_audit.csv", index=False)
    report = classification_report(test["label"], predicted, labels=[0, 1],
                                   output_dict=True, zero_division=0)
    matrix = confusion_matrix(test["label"], predicted, labels=[0, 1]).tolist()
    metrics = {
        "model": MODEL, "revision": REVISION, "gpu": torch.cuda.get_device_name(0),
        "pytorch": torch.__version__, "cuda": torch.version.cuda, "seed": seed,
        "trainable_parameters": count, "parameters": sum(p.numel() for p in model.parameters()),
        "epochs": EPOCHS, "best_epoch": best_epoch, "max_length": MAX_LENGTH,
        "micro_batch": MICRO_BATCH, "gradient_accumulation": ACCUMULATION,
        "gradient_normalization": "sample_weighted_actual_group_size",
        "learning_rate": LEARNING_RATE,
        "peak_vram_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
        "wall_seconds": round(time.perf_counter() - start, 2),
        "counts": {name: item["retained"] for name, item in audit.items()},
        "test_macro_f1": report["macro avg"]["f1-score"], "test_accuracy": report["accuracy"],
        "complaint_precision": report["1"]["precision"], "complaint_recall": report["1"]["recall"],
        "complaint_f1": report["1"]["f1-score"], "confusion_matrix": matrix, "history": history,
        "test_rows": len(test),
    }
    (folder / "phobert_results.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    tokenizer.save_pretrained(folder / "tokenizer")
    model.config.save_pretrained(folder / "config")
    pd.DataFrame(history).to_csv(folder / "training_history.csv", index=False)
    pd.DataFrame({"test_row": test.index, "label": test["label"], "prediction": predicted,
                  "domain": test["domain"]}).to_csv(folder / "phobert_predictions.csv", index=False)
    torch.save({name: value.cpu() for name, value in trainable.items()},
               folder / "trainable_weights.pt")
    print(json.dumps(metrics, indent=2), flush=True)
    return metrics


In [ ]:
from pathlib import Path
import gc
import torch

gpu_results = []
for seed in SEEDS:
    result = fine_tune(seed=seed, results_dir=f"results_phobert/seed_{seed}")
    gpu_results.append(result)
    del result
    gc.collect()
    torch.cuda.empty_cache()

from importlib.metadata import PackageNotFoundError, version
import platform
import sys

def package_version(distribution):
    try:
        return version(distribution)
    except PackageNotFoundError:
        return None

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {name: package_version(name) for name in (
        "numpy", "pandas", "scikit-learn", "underthesea", "matplotlib",
        "transformers", "tokenizers", "torch",
    )},
    "cuda_available": torch.cuda.is_available(),
    "cuda_runtime": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "dataset_commit": "25e1851683ebc407d28e5a2d514b7ae0eadd4e4b",
    "model": MODEL,
    "model_revision": REVISION,
    "seeds": list(SEEDS),
}
Path("results_phobert/environment.json").write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(environment, ensure_ascii=False, indent=2))


In [ ]:
import gc
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer

# Release a model cached by a prior execution before restoring checkpoints for verification.
_DEMO_CHECKPOINT = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def load_saved_checkpoint(checkpoint_dir):
    checkpoint_dir = Path(checkpoint_dir)
    metadata = json.loads((checkpoint_dir / "phobert_results.json").read_text(encoding="utf-8"))
    config = AutoConfig.from_pretrained(checkpoint_dir / "config")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir / "tokenizer", use_fast=False)
    model = AutoModelForSequenceClassification.from_pretrained(
        metadata["model"], revision=metadata["revision"], config=config)
    model.requires_grad_(False)
    model.roberta.encoder.layer[-2:].requires_grad_(True)
    model.classifier.requires_grad_(True)
    saved = torch.load(checkpoint_dir / "trainable_weights.pt",
                       map_location="cpu", weights_only=True)
    trainable_names = {name for name, parameter in model.named_parameters()
                       if parameter.requires_grad}
    if set(saved) != trainable_names:
        raise ValueError("Checkpoint trainable parameter names do not match this model revision.")
    missing, unexpected = model.load_state_dict(saved, strict=False)
    expected_missing = set(model.state_dict()) - set(saved)
    if set(missing) != expected_missing or unexpected:
        raise ValueError("Checkpoint state did not restore cleanly.")
    model.to("cuda").eval()
    return model, tokenizer, metadata

def make_test_loader(test_frame, tokenizer, max_length, micro_batch):
    texts = test_frame["text"].map(lambda value: word_tokenize(value, format="text")).tolist()
    encodings = tokenizer(texts, max_length=max_length, truncation=True)
    items = [{**{key: values[i] for key, values in encodings.items()},
              "labels": int(label)}
             for i, label in enumerate(test_frame["label"])]
    return DataLoader(items, batch_size=micro_batch, shuffle=False, num_workers=0,
                      pin_memory=True, collate_fn=lambda rows: tokenizer.pad(
                          rows, padding=True, pad_to_multiple_of=8, return_tensors="pt"))

test_frame, _ = load_splits()
test_frame = test_frame["test"]
restoration_rows = []
for seed in SEEDS:
    checkpoint = Path(f"results_phobert/seed_{seed}")
    restored_model, restored_tokenizer, metadata = load_saved_checkpoint(checkpoint)
    restored_predictions = predict(
        restored_model,
        make_test_loader(test_frame, restored_tokenizer,
                         metadata["max_length"], metadata["micro_batch"]))
    saved_predictions = pd.read_csv(checkpoint / "phobert_predictions.csv")
    keys = ["test_row", "label", "domain"]
    expected_keys = pd.DataFrame({
        "test_row": test_frame.index, "label": test_frame["label"].to_numpy(),
        "domain": test_frame["domain"].to_numpy(),
    })
    if not expected_keys["test_row"].is_unique:
        raise ValueError("Current test split contains duplicate row IDs.")
    if len(saved_predictions) != len(test_frame) or not saved_predictions["test_row"].is_unique:
        raise ValueError(f"Saved prediction row count/IDs are invalid for seed {seed}.")
    expected_sorted = expected_keys.sort_values("test_row").reset_index(drop=True)
    saved_predictions = saved_predictions.sort_values("test_row").reset_index(drop=True)
    if not saved_predictions[keys].equals(expected_sorted[keys]):
        raise ValueError(f"Saved prediction IDs/labels/domains differ for seed {seed}.")
    restored_frame = expected_keys.assign(
        prediction_restored=np.asarray(restored_predictions)
    ).sort_values("test_row").reset_index(drop=True)
    expected_keys = expected_sorted
    aligned = saved_predictions.merge(
        restored_frame, on=keys, how="inner", validate="one_to_one", sort=False)
    exact_match = (
        len(aligned) == len(test_frame)
        and np.array_equal(aligned["test_row"].to_numpy(), expected_keys["test_row"].to_numpy())
        and np.array_equal(aligned["prediction"].to_numpy(),
                           aligned["prediction_restored"].to_numpy())
    )
    if not exact_match:
        raise AssertionError(f"Restored checkpoint predictions differ for seed {seed}.")
    restoration_rows.append({
        "seed": seed, "test_rows": len(aligned),
        "predictions_identical_to_saved": bool(exact_match),
        "base_model_revision": metadata["revision"],
        "max_length": metadata["max_length"], "micro_batch": metadata["micro_batch"],
    })
    del restored_model, restored_tokenizer
    torch.cuda.empty_cache()
Path("results_phobert/checkpoint_restore_verification.json").write_text(
    json.dumps(restoration_rows, indent=2), encoding="utf-8")
display(pd.DataFrame(restoration_rows))


def _load_demo_checkpoint():
    """Restore seed 42 once so repeated demo predictions reuse the checkpoint."""
    global _DEMO_CHECKPOINT
    if _DEMO_CHECKPOINT is None:
        _DEMO_CHECKPOINT = load_saved_checkpoint("results_phobert/seed_42")
    return _DEMO_CHECKPOINT


def release_demo_checkpoint():
    """Release the cached demo model and its GPU memory."""
    global _DEMO_CHECKPOINT
    if _DEMO_CHECKPOINT is not None:
        _DEMO_CHECKPOINT = None
        gc.collect()
        torch.cuda.empty_cache()


def predict_phobert_review(review):
    """Classify one non-empty Vietnamese review with the seed-42 checkpoint."""
    if not isinstance(review, str):
        raise TypeError("review phải là chuỗi văn bản.")
    normalized = normalize(review)
    if not normalized.strip():
        raise ValueError("Vui lòng nhập review, không để trống hoặc chỉ có khoảng trắng.")
    text = word_tokenize(normalized, format="text")
    if not text.strip():
        raise ValueError("Không còn nội dung sau chuẩn hóa và tách từ.")
    if not torch.cuda.is_available():
        raise RuntimeError("Demo PhoBERT cần chạy trên môi trường CUDA/Kaggle GPU.")

    model, tokenizer, metadata = _load_demo_checkpoint()
    max_length = int(metadata["max_length"])
    untruncated = tokenizer(text, add_special_tokens=True, truncation=False)
    input_token_count = len(untruncated["input_ids"])
    encoded = tokenizer(
        text, max_length=max_length, truncation=True, return_tensors="pt"
    )
    used_token_count = int(encoded["input_ids"].shape[1])
    was_truncated = input_token_count > used_token_count
    encoded = {key: value.to("cuda") for key, value in encoded.items()}
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        logits = model(**encoded).logits.float()
    probabilities = torch.softmax(logits, dim=-1)[0].cpu().tolist()
    predicted = int(np.argmax(probabilities))
    display_names = {0: "không có lời phàn nàn", 1: "có lời phàn nàn"}
    return {
        "seed": int(metadata["seed"]),
        "label": predicted,
        "prediction": display_names[predicted],
        "softmax_score": float(probabilities[predicted]),
        "probabilities": {display_names[label]: float(score) for label, score in enumerate(probabilities)},
        "input_token_count": input_token_count,
        "used_token_count": used_token_count,
        "max_length": max_length,
        "truncated": was_truncated,
    }


In [ ]:
import json
from pathlib import Path
import pandas as pd

baseline = pd.read_csv("results/metrics.csv")
seed_table = pd.DataFrame([{
    "seed": r["seed"], "best_epoch": r["best_epoch"], "gpu": r["gpu"],
    "val_macro_f1": max(e["val_macro_f1"] for e in r["history"]),
    "test_macro_f1": r["test_macro_f1"], "test_accuracy": r["test_accuracy"],
    "complaint_precision": r["complaint_precision"],
    "complaint_recall": r["complaint_recall"],
    "complaint_f1": r["complaint_f1"], "wall_seconds": r["wall_seconds"],
    "peak_vram_mib": r["peak_vram_mib"],
} for r in gpu_results])
out = Path("results_phobert")
seed_table.to_csv(out / "seed_metrics.csv", index=False)
summary = {
    "seeds": list(SEEDS), "n_runs": len(gpu_results),
    "test_macro_f1_mean": float(seed_table["test_macro_f1"].mean()),
    "test_macro_f1_sd_sample": float(seed_table["test_macro_f1"].std(ddof=1)),
    "test_accuracy_mean": float(seed_table["test_accuracy"].mean()),
    "test_accuracy_sd_sample": float(seed_table["test_accuracy"].std(ddof=1)),
    "best_epochs": seed_table["best_epoch"].astype(int).tolist(),
    "gpu_models": sorted(seed_table["gpu"].unique().tolist()),
    "interpretation": "controlled repeat on previously inspected fixed test set; not independent confirmation",
}
(out / "aggregate_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
baseline_predictions = pd.read_csv("results/baseline_test_predictions.csv")
for seed in SEEDS:
    model_predictions = pd.read_csv(f"results_phobert/seed_{seed}/phobert_predictions.csv")
    if not baseline_predictions["test_row"].equals(model_predictions["test_row"]):
        raise ValueError(f"Test row IDs differ for seed {seed}")
    if not baseline_predictions["label"].equals(model_predictions["label"]):
        raise ValueError(f"Test labels differ for seed {seed}")
comparison = pd.concat([
    baseline,
    pd.DataFrame([{
        "configuration": "D_PhoBERT_3seed_mean",
        "val_macro_f1": float(seed_table["val_macro_f1"].mean()),
        "val_macro_f1_sd": float(seed_table["val_macro_f1"].std(ddof=1)),
        "test_macro_f1": summary["test_macro_f1_mean"],
        "test_macro_f1_sd": summary["test_macro_f1_sd_sample"],
        "test_accuracy": summary["test_accuracy_mean"],
        "test_accuracy_sd": summary["test_accuracy_sd_sample"],
        "complaint_precision": float(seed_table["complaint_precision"].mean()),
        "complaint_recall": float(seed_table["complaint_recall"].mean()),
        "complaint_f1": float(seed_table["complaint_f1"].mean()),
    }])
], ignore_index=True, sort=False)
comparison.to_csv(out / "comparison.csv", index=False)
display(seed_table.round(4))
display(pd.DataFrame([summary]))

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

baseline_all = pd.read_csv("results/baseline_all_test_predictions.csv")
if not baseline_all["test_row"].is_unique:
    raise ValueError("Baseline per-domain table contains duplicate test rows.")
per_domain_rows = []
seed_predictions_by_seed = {}

def append_domain_metrics(model_name, seed, frame, prediction_column):
    for domain, subset in frame.groupby("domain", sort=True):
        y_true = subset["label"].to_numpy()
        y_pred = subset[prediction_column].to_numpy()
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        per_domain_rows.append({
            "model": model_name, "seed": seed, "domain": domain, "n": len(subset),
            "macro_f1": f1_score(y_true, y_pred, labels=[0, 1], average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred),
            "complaint_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            "complaint_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            "complaint_f1": f1_score(y_true, y_pred, pos_label=1, average="binary", zero_division=0),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        })

baseline_columns = {
    "A_LogReg_space": "prediction_A_LogReg_space",
    "B_SVM_space": "prediction_B_SVM_space",
    "C_SVM_segmented": "prediction_C_SVM_segmented",
}
for model_name, prediction_column in baseline_columns.items():
    if prediction_column not in baseline_all:
        raise ValueError(f"Missing baseline test predictions for {model_name}.")
    append_domain_metrics(model_name, "", baseline_all, prediction_column)
for seed in SEEDS:
    frame = pd.read_csv(f"results_phobert/seed_{seed}/phobert_predictions.csv")
    if not frame["test_row"].is_unique or len(frame) != len(baseline_all):
        raise ValueError(f"Invalid domain-evaluation rows for PhoBERT seed {seed}.")
    aligned = baseline_all[["test_row", "label", "domain"]].merge(
        frame, on=["test_row", "label", "domain"], how="inner", validate="one_to_one", sort=False
    )
    if len(aligned) != len(baseline_all):
        raise ValueError(f"PhoBERT seed {seed} does not align with the baseline test rows.")
    append_domain_metrics(f"D_PhoBERT_seed_{seed}", int(seed), aligned, "prediction")
    seed_predictions_by_seed[int(seed)] = aligned["prediction"].to_numpy()
per_domain = pd.DataFrame(per_domain_rows)
per_domain.to_csv("results_phobert/per_domain_metrics.csv", index=False)
display(per_domain.round(4))


## Phân tích chuyển tiếp lỗi và độ bất định theo miền

Các bảng so sánh cùng ID test cho baseline C và PhoBERT ở từng seed: PhoBERT sửa lỗi nào, tạo thêm lỗi nào, và lỗi nào cả hai cùng mắc. Khoảng bootstrap ghép cặp theo miền là phân tích mô tả hậu nghiệm trên test cố định; không dùng để chọn mô hình và không phải xác nhận độc lập.


In [ ]:
# Post-hoc paired error transitions versus baseline C on the previously inspected test set.
if not baseline_all["test_row"].is_unique:
    raise ValueError("Baseline test_row IDs are not unique for paired transition analysis.")
expected_test_rows = baseline_all["test_row"].to_numpy()
transition_rows = []
for seed in SEEDS:
    frame = pd.read_csv(f"results_phobert/seed_{seed}/phobert_predictions.csv")
    aligned = baseline_all[["test_row", "label", "domain", "prediction_C_SVM_segmented"]].merge(
        frame, on=["test_row", "label", "domain"], how="inner", validate="one_to_one", sort=False
    )
    if len(aligned) != len(baseline_all) or not np.array_equal(
            aligned["test_row"].to_numpy(), expected_test_rows):
        raise ValueError(f"PhoBERT seed {seed} does not exactly align to baseline C test row order.")
    baseline_correct = aligned["prediction_C_SVM_segmented"].eq(aligned["label"])
    phobert_correct = aligned["prediction"].eq(aligned["label"])
    row = aligned.rename(columns={
        "label": "gold_label", "prediction_C_SVM_segmented": "baseline_c_prediction",
        "prediction": "phobert_prediction",
    })
    row["seed"] = int(seed)
    row["baseline_error_type"] = np.where(
        baseline_correct, "correct",
        np.where(row["gold_label"].eq(0), "false_positive", "false_negative"),
    )
    row["phobert_error_type"] = np.where(
        phobert_correct, "correct",
        np.where(row["gold_label"].eq(0), "false_positive", "false_negative"),
    )
    row["transition"] = np.select(
        [~baseline_correct & phobert_correct,
         baseline_correct & ~phobert_correct,
         ~baseline_correct & ~phobert_correct],
        ["phobert_fixes_baseline", "phobert_introduces_error", "both_wrong"],
        default="both_correct",
    )
    transition_rows.append(row[[
        "test_row", "gold_label", "domain", "seed", "baseline_c_prediction",
        "phobert_prediction", "baseline_error_type", "phobert_error_type", "transition",
    ]])
transitions = pd.concat(transition_rows, ignore_index=True)
if len(transitions) != len(baseline_all) * len(SEEDS):
    raise ValueError("Paired error transition table has an unexpected row count.")
transitions.to_csv("results_phobert/paired_error_transitions.csv", index=False)
transition_counts = transitions.groupby(
    ["seed", "domain", "gold_label", "transition"], as_index=False
).size().rename(columns={"size": "count"})
transition_denominators = transitions.groupby(
    ["seed", "domain", "gold_label"], as_index=False
).size().rename(columns={"size": "denominator"})
transition_summary = transition_counts.merge(
    transition_denominators, on=["seed", "domain", "gold_label"], validate="many_to_one"
)
transition_summary["proportion"] = transition_summary["count"] / transition_summary["denominator"]
transition_summary.to_csv("results_phobert/paired_error_transition_summary.csv", index=False)

text_lookup = test_frame[["text"]].copy()
text_lookup.index.name = "test_row"
text_lookup = text_lookup.reset_index()
seed42_errors = transitions.loc[
    transitions["seed"].eq(42) & transitions["phobert_error_type"].ne("correct")
].rename(columns={
    "gold_label": "label", "phobert_prediction": "predicted",
}).merge(text_lookup, on="test_row", how="left", validate="one_to_one")
if seed42_errors["text"].isna().any():
    raise ValueError("Could not align review text to the bounded seed-42 error sample.")
seed42_errors["text"] = seed42_errors["text"].astype(str).str.slice(0, 180)
phobert_error_sample = select_error_examples(seed42_errors, per_label=5, random_seed=42)
phobert_error_sample = phobert_error_sample.rename(columns={"text": "text_excerpt"})
phobert_error_sample[[
    "test_row", "seed", "label", "predicted", "domain", "text_excerpt",
]].to_csv("results_phobert/phobert_error_examples_seed42.csv", index=False)

# Six fixed, post-hoc cases illustrate transitions; explanations are hypotheses, not causes.
case_notes = {
    126: ("phobert_fixes_baseline", "Câu hỏi về bảo mật, không nêu bất mãn trực tiếp.",
          "PhoBERT có thể phân biệt câu hỏi thông tin với khiếu nại tốt hơn baseline trong ca này."),
    512: ("phobert_fixes_baseline", "Nêu ứng dụng nặng, không chơi được và tự thoát sau bảo trì.",
          "Chuỗi hệ quả tiêu cực có thể giúp mô hình nhận diện khiếu nại theo ngữ cảnh."),
    53: ("phobert_introduces_error", "Lặp câu hỏi tu từ nhưng đồng thời khen game hay và sáng tạo.",
         "Câu hỏi lặp hoặc giọng cường điệu có thể gợi nhầm nhãn phàn nàn dù nhãn corpus là 0."),
    164: ("phobert_introduces_error", "Câu rất ngắn với từ đánh giá tiêu cực trực tiếp: ‘tệ’, ‘lừa gạt’.",
          "Ca này chưa giải thích được; cần kiểm tra độ bao phủ cụm từ hoặc chất lượng nhãn."),
    52: ("both_wrong", "Nêu việc chưa có máy chủ, sau đó nói game thú vị và hứa đánh giá 5 sao.",
         "Sự pha trộn giữa dấu hiệu bất mãn và đánh giá tích cực có thể gây nhầm; nhãn biên cần được xem lại."),
    195: ("both_wrong", "Câu ngắn, diễn đạt bất thường nên ý nghĩa khó xác định.",
          "Có thể có vấn đề về độ rõ của văn bản hoặc nhãn; không thể suy ra nguyên nhân chỉ từ ví dụ."),
}
case_ids = list(case_notes)
case_frame = transitions.loc[
    transitions["seed"].eq(42) & transitions["test_row"].isin(case_ids)
].copy()
if set(case_frame["test_row"].astype(int)) != set(case_ids) or case_frame["test_row"].duplicated().any():
    raise ValueError("Fixed qualitative cases do not match the current test predictions.")
expected_transitions = {row_id: values[0] for row_id, values in case_notes.items()}
actual_transitions = dict(zip(case_frame["test_row"].astype(int), case_frame["transition"]))
if actual_transitions != expected_transitions:
    raise ValueError("A fixed qualitative case changed transition type; review its note before proceeding.")
case_frame = case_frame.merge(text_lookup, on="test_row", how="left", validate="one_to_one")
case_frame["text_excerpt"] = case_frame["text"].astype(str).str.slice(0, 180)
case_frame["observed_pattern"] = case_frame["test_row"].map(lambda row_id: case_notes[int(row_id)][1])
case_frame["interpretation_hypothesis"] = case_frame["test_row"].map(lambda row_id: case_notes[int(row_id)][2])
case_frame["interpretation_limit"] = (
    "One-pass post-hoc qualitative note; not independently validated and not a causal finding."
)
case_frame[[
    "test_row", "gold_label", "domain", "baseline_c_prediction", "phobert_prediction",
    "transition", "text_excerpt", "observed_pattern", "interpretation_hypothesis", "interpretation_limit",
]].sort_values("test_row").to_csv(
    "results_phobert/paired_error_case_studies_seed42.csv", index=False
)

# Domain-wise paired bootstrap; intervals are descriptive and conditional on this fixed test set.
N_DOMAIN_BOOTSTRAP = 2000
def macro_f1_replicates(y_values, predictions):
    tp = ((predictions == 1) & (y_values == 1)).sum(axis=-1)
    tn = ((predictions == 0) & (y_values == 0)).sum(axis=-1)
    fp = ((predictions == 1) & (y_values == 0)).sum(axis=-1)
    fn = ((predictions == 0) & (y_values == 1)).sum(axis=-1)
    f1_pos = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    f1_neg = 2 * tn / np.maximum(2 * tn + fp + fn, 1)
    return (f1_pos + f1_neg) / 2

domain_bootstrap_rows = []
for domain_index, domain_name in enumerate(sorted(baseline_all["domain"].unique())):
    subset = baseline_all.loc[baseline_all["domain"].eq(domain_name)]
    y_domain = subset["label"].to_numpy()
    p0_domain = subset["prediction_C_SVM_segmented"].to_numpy()
    seed_domain = np.stack([
        seed_predictions_by_seed[int(seed)][baseline_all["domain"].eq(domain_name).to_numpy()]
        for seed in SEEDS
    ])
    strata = [np.flatnonzero(y_domain == label) for label in LABELS]
    if any(len(ids) == 0 for ids in strata):
        raise ValueError(f"Domain {domain_name} lacks a class required for stratified bootstrap.")
    bootstrap_seed = 20261112 + domain_index
    rng = np.random.default_rng(bootstrap_seed)
    samples = np.concatenate([
        rng.choice(ids, size=(N_DOMAIN_BOOTSTRAP, len(ids)), replace=True) for ids in strata
    ], axis=1)
    y_replicates = y_domain[samples]
    baseline_replicates = macro_f1_replicates(y_replicates, p0_domain[samples])
    seed_replicates = macro_f1_replicates(y_replicates[None, :, :], seed_domain[:, samples])
    mean_seed_replicates = seed_replicates.mean(axis=0)
    difference_replicates = mean_seed_replicates - baseline_replicates
    lower, upper = np.quantile(difference_replicates, [0.025, 0.975])
    observed_difference = float(np.mean([
        f1_score(y_domain, prediction, labels=LABELS, average="macro", zero_division=0)
        for prediction in seed_domain
    ]) - f1_score(y_domain, p0_domain, labels=LABELS, average="macro", zero_division=0))
    domain_bootstrap_rows.append({
        "domain": domain_name, "n": len(subset),
        "class_0_n": int(np.sum(y_domain == 0)), "class_1_n": int(np.sum(y_domain == 1)),
        "baseline_c_macro_f1": float(f1_score(
            y_domain, p0_domain, labels=LABELS, average="macro", zero_division=0)),
        "phobert_mean_seed_macro_f1": float(np.mean([
            f1_score(y_domain, prediction, labels=LABELS, average="macro", zero_division=0)
            for prediction in seed_domain
        ])),
        "observed_difference": observed_difference,
        "ci95_lower": float(lower), "ci95_upper": float(upper),
        "bootstrap_samples": N_DOMAIN_BOOTSTRAP, "bootstrap_seed": bootstrap_seed,
        "interpretation": "descriptive paired interval conditional on the previously inspected fixed test set; no multiplicity correction",
    })
pd.DataFrame(domain_bootstrap_rows).to_csv(
    "results_phobert/per_domain_paired_bootstrap.csv", index=False
)
Path("results_phobert/paired_error_transition_metadata.json").write_text(
    json.dumps({
        "test_rows": int(len(baseline_all)), "paired_rows": int(len(transitions)),
        "seeds": [int(seed) for seed in SEEDS], "baseline": "C_SVM_segmented",
        "error_sample_seed": 42, "error_sample_max_per_class": 5,
        "error_sample_method": "largest-remainder proportional allocation by domain",
        "transition_table_contains_review_text": False,
        "bounded_error_examples_seed42": int(len(phobert_error_sample)),
        "qualitative_case_ids_seed42": case_ids,
        "qualitative_note_limit": "one-pass post-hoc hypotheses; not independently validated or causal",
        "interpretation": "post-hoc descriptive analysis on a previously inspected fixed test set; not used for model selection or tuning",
    }, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(transition_summary.groupby(["seed", "transition"])["count"].sum().unstack(fill_value=0).to_string())



## So sánh ghép cặp với baseline được validation chọn

Bootstrap phân tầng, lấy lại mẫu cùng các ID cho baseline và cả ba mô hình PhoBERT trong mỗi lần lặp. Mỗi lần lặp tính Macro-F1 riêng cho từng seed rồi lấy trung bình ba seed; chênh lệch được tính so với cấu hình baseline được chọn trên validation. Khoảng tin cậy có điều kiện trên ba mô hình đã huấn luyện và test đã xem ở vòng trước; đây không phải xác nhận độc lập.


In [ ]:
"""Paired stratified bootstrap for the baseline versus the mean of three PhoBERT seeds."""
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED_BOOTSTRAP = 20261112
N_BOOTSTRAP = 2000
base = pd.read_csv("results/baseline_test_predictions.csv")
seed_frames = [pd.read_csv(f"results_phobert/seed_{seed}/phobert_predictions.csv")
               for seed in SEEDS]
keys = ["test_row", "label", "domain"]
for frame in seed_frames:
    joined = base.merge(frame, on=keys, how="inner", validate="one_to_one")
    if len(joined) != len(base) or len(joined) != len(frame):
        raise ValueError("A PhoBERT seed does not contain exactly the baseline test rows.")
    if not np.array_equal(joined["test_row"].to_numpy(), base["test_row"].to_numpy()):
        raise ValueError("Test row order/IDs mismatch after join.")
y = base["label"].to_numpy()
p0 = base["prediction_baseline"].to_numpy()
if not base["test_row"].is_unique:
    raise ValueError("Baseline test_row IDs are not unique.")
if not set(np.unique(y)).issubset({0, 1}):
    raise ValueError("The bootstrap expects binary labels 0 and 1.")
aligned_predictions = []
for frame in seed_frames:
    joined = base[keys].merge(
        frame, on=keys, how="inner", validate="one_to_one", sort=False)
    if len(joined) != len(base) or not np.array_equal(
            joined["test_row"].to_numpy(), base["test_row"].to_numpy()):
        raise ValueError("Could not align every seed prediction to baseline test IDs.")
    aligned_predictions.append(joined["prediction"].to_numpy())
ps = np.stack(aligned_predictions)
if ps.shape != (len(SEEDS), len(y)):
    raise ValueError(f"Unexpected prediction matrix shape {ps.shape}")
rng = np.random.default_rng(SEED_BOOTSTRAP)
strata = [np.flatnonzero(y == label) for label in (0, 1)]
samples = np.concatenate([rng.choice(ids, size=(N_BOOTSTRAP, len(ids)), replace=True)
                          for ids in strata], axis=1)
y_rep = y[samples]
p0_rep = p0[samples]
ps_rep = ps[:, samples]

base_boot = macro_f1_replicates(y_rep, p0_rep)
seed_boot = macro_f1_replicates(y_rep[None, :, :], ps_rep)
mean_seed_boot = seed_boot.mean(axis=0)
observed = float(np.mean([r["test_macro_f1"] for r in gpu_results]) -
                 macro_f1_replicates(y[None, :], p0[None, :])[0])
difference_boot = mean_seed_boot - base_boot
lo, hi = np.quantile(difference_boot, [0.025, 0.975])
baseline_metadata = json.loads(Path("results/summary.json").read_text(encoding="utf-8"))
paired_summary = {
    "baseline_configuration": baseline_metadata["selected"],
    "test_rows": int(len(base)), "baseline_macro_f1": float(macro_f1_replicates(y[None, :], p0[None, :])[0]),
    "phobert_mean_seed_macro_f1": float(np.mean([r["test_macro_f1"] for r in gpu_results])),
    "observed_mean_difference": observed, "paired_bootstrap_ci95": [float(lo), float(hi)],
    "bootstrap_samples": N_BOOTSTRAP, "bootstrap_seed": SEED_BOOTSTRAP,
    "seeds": list(SEEDS), "same_test_row_ids_for_all_models": True,
    "interpretation": "conditional paired interval for the mean of three trained seeds on the previously inspected fixed test set",
}
Path("results_phobert/paired_bootstrap.json").write_text(
    json.dumps(paired_summary, indent=2), encoding="utf-8")
pd.DataFrame({"test_row": base["test_row"], "label": y, "domain": base["domain"],
              "prediction_baseline": p0,
              **{f"prediction_seed_{seed}": ps[i] for i, seed in enumerate(SEEDS)}
             }).to_csv("results_phobert/all_test_predictions.csv", index=False)
display(pd.DataFrame([paired_summary]))


## Thử nghiệm tăng cường dữ liệu

Giữ nguyên kích thước và nhãn tập huấn luyện. F0 lặp bản gốc, F1 dùng bản bỏ dấu cho nửa số lượt mẫu. Kết quả là thăm dò trên bản chia hiện có.

In [ ]:
"""Compare paired test predictions from the classical baseline and PhoBERT."""
import json
from pathlib import Path

import numpy as np
import pandas as pd


def paired_comparison(classical_file, gpu_file):
    classical = pd.read_csv(classical_file)
    gpu = pd.read_csv(gpu_file)
    columns = ["test_row", "label", "domain"]
    joined = classical.merge(gpu, on=columns, how="inner", validate="one_to_one")
    if len(joined) != len(classical) or len(joined) != len(gpu):
        raise ValueError("The two sets of test predictions must contain exactly the same rows.")
    y = joined["label"].to_numpy()
    p = joined[["prediction_c", "prediction"]].to_numpy().T
    correct = p == y
    repaired = ~correct[0] & correct[1]
    introduced = correct[0] & ~correct[1]

    # A paired, stratified bootstrap describes uncertainty on this fixed test set.
    rng = np.random.default_rng(42)
    strata = [np.flatnonzero(y == label) for label in (0, 1)]
    samples = np.concatenate(
        [rng.choice(ids, size=(2000, len(ids)), replace=True) for ids in strata], axis=1
    )
    y_sample = y[samples][None, :, :]
    p_sample = p[:, samples]
    tp = ((p_sample == 1) & (y_sample == 1)).sum(axis=2)
    tn = ((p_sample == 0) & (y_sample == 0)).sum(axis=2)
    fp = ((p_sample == 1) & (y_sample == 0)).sum(axis=2)
    fn = ((p_sample == 0) & (y_sample == 1)).sum(axis=2)
    f1_pos = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    f1_neg = 2 * tn / np.maximum(2 * tn + fp + fn, 1)
    macro = (f1_pos + f1_neg) / 2
    lo, hi = np.quantile(macro[1] - macro[0], [.025, .975])
    return {
        "test_rows": len(joined), "c_wrong_d_right": int(repaired.sum()),
        "c_right_d_wrong": int(introduced.sum()), "both_wrong": int((~correct).all(axis=0).sum()),
        "bootstrap_difference_ci95": [float(lo), float(hi)], "bootstrap_samples": 2000,
        "c_wrong_d_right_ids": joined.loc[repaired, "test_row"].head(5).tolist(),
        "c_right_d_wrong_ids": joined.loc[introduced, "test_row"].head(5).tolist(),
    }

"""Exploratory ablation of accent-removal augmentation on a fixed model."""
import json
from pathlib import Path

import numpy as np
import pandas as pd



def run_augmentation(folder):
    splits, _ = load_splits()
    train, val, test = (splits[k] for k in ("train", "val", "test"))
    clean = {k:segmented(df["text"]) for k,df in splits.items()}
    unaccented = {k:segmented(df["text"].map(remove_accents)) for k,df in splits.items()}
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    train_unaccented = unaccented["train"]
    test_unaccented = unaccented["test"]
    train_label_map = {}
    for text, label in zip(train_unaccented, train["label"]):
        train_label_map.setdefault(text, set()).add(int(label))
    overlap = pd.Series(
        [text in train_label_map for text in test_unaccented], index=test.index)
    sensitivity = test.loc[~overlap]
    sensitivity_unaccented = segmented(sensitivity["text"].map(remove_accents))
    rows, predictions = [], {}
    for name, second_view in (("F0_clean_duplicate", clean["train"]),
                              ("F1_accent_augmentation", unaccented["train"])):
        # Match the corpus size, label counts, and feature/hyperparameter settings.
        X = clean["train"] + second_view
        y = pd.concat([train["label"], train["label"]], ignore_index=True)
        model = build_model(True).fit(X, y)
        predictions[name] = {
            "clean": model.predict(clean["test"]),
            "unaccented": model.predict(unaccented["test"]),
            "unaccented_sensitivity": model.predict(sensitivity_unaccented),
        }
        rows.append({"model":name,"training_rows":len(X),
                     "val_clean":score(val["label"],model.predict(clean["val"])),
                     "val_unaccented":score(val["label"],model.predict(unaccented["val"])),
                     "test_clean":score(test["label"],predictions[name]["clean"]),
                     "test_unaccented":score(test["label"],predictions[name]["unaccented"]),
                     "test_unaccented_sensitivity":score(
                         sensitivity["label"],predictions[name]["unaccented_sensitivity"]),
                     "sensitivity_test_n":int(len(sensitivity)),
                     "accent_overlap_test_n":int(overlap.sum())})
    comparisons = {}
    for condition in ("clean", "unaccented", "unaccented_sensitivity"):
        left, right = (predictions[row["model"]][condition] for row in rows)
        evaluation = sensitivity if condition == "unaccented_sensitivity" else test
        metadata = {
            "test_row": evaluation.index, "label": evaluation["label"].to_numpy(),
            "domain": evaluation["domain"].to_numpy()
        }
        baseline_path = folder / f"augmentation_{condition}_baseline.csv"
        augmented_path = folder / f"augmentation_{condition}_augmented.csv"
        pd.DataFrame({**metadata,"prediction_c":left}).to_csv(baseline_path,index=False)
        pd.DataFrame({**metadata,"prediction":right}).to_csv(augmented_path,index=False)
        comparison = paired_comparison(baseline_path, augmented_path)
        comparison["macro_f1_difference"] = score(evaluation["label"], right) - score(evaluation["label"], left)
        comparison["correct_difference"] = int(
            np.sum(right == evaluation["label"]) - np.sum(left == evaluation["label"]))
        comparison["evaluation_rows"] = int(len(evaluation))
        comparisons[condition] = comparison
    (folder/"accent_augmentation.json").write_text(json.dumps(rows, indent=2), encoding="utf-8")
    (folder/"accent_augmentation_uncertainty.json").write_text(json.dumps(comparisons, indent=2), encoding="utf-8")
    print(pd.DataFrame(rows).round(4).to_string(index=False))
    print(json.dumps(comparisons, indent=2))

In [ ]:
run_augmentation("results_robustness")

## Đánh giá thăm dò độ bền PhoBERT theo ba seed

Phần này đánh giá lại ba checkpoint đã chọn bằng validation trên cùng test, bản biến đổi bỏ dấu và 18 cặp chẩn đoán tự soạn. Đây là phân tích hậu nghiệm trên corpus cố định, không phải tập ngoài miền độc lập, không dùng để chọn mô hình. Bốn mẫu test có va chạm sau bỏ dấu với train được giữ trong kết quả chính và loại riêng trong phân tích sensitivity.


In [ ]:
"""Post-hoc robustness evaluation of the three saved PhoBERT checkpoints."""
import gc
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

splits_robust, _ = load_splits()
train_robust, test_robust = splits_robust["train"], splits_robust["test"]
train_unaccented_seg = segmented(train_robust["text"].map(remove_accents))
test_unaccented_texts = test_robust["text"].map(remove_accents).tolist()
test_unaccented_seg = segmented(test_unaccented_texts)
train_view = set(train_unaccented_seg)
overlap_mask_robust = np.array([text in train_view for text in test_unaccented_seg])
sensitivity_frame = test_robust.loc[~overlap_mask_robust].copy()
sensitivity_unaccented_texts = [text for text, keep in zip(test_unaccented_texts, ~overlap_mask_robust) if keep]
challenge_frame = pd.DataFrame({
    "text": [sentence for _, complaint, ordinary in PAIRS for sentence in (complaint, ordinary)],
    "label": [label for _ in PAIRS for label in (1, 0)],
    "domain": ["synthetic"] * (2 * len(PAIRS)),
})
challenge_texts = challenge_frame["text"].tolist()

def metric_row(y, prediction):
    return {
        "macro_f1": float(f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y, prediction)),
        "complaint_precision": float(precision_score(y, prediction, pos_label=1, zero_division=0)),
        "complaint_recall": float(recall_score(y, prediction, pos_label=1, zero_division=0)),
    }

robust_rows, pair_rows = [], []
robust_out = Path("results_phobert")
for seed in SEEDS:
    checkpoint = Path(f"results_phobert/seed_{seed}")
    restored_model, restored_tokenizer, metadata = load_saved_checkpoint(checkpoint)
    def predict_texts(items):
        frame = pd.DataFrame({"text": items, "label": np.zeros(len(items), dtype=int),
                              "domain": ["robustness"] * len(items)})
        return predict(restored_model, make_test_loader(
            frame, restored_tokenizer, metadata["max_length"], metadata["micro_batch"]))
    unaccented_pred = np.asarray(predict_texts(test_unaccented_texts))
    sensitivity_pred = np.asarray(predict_texts(sensitivity_unaccented_texts))
    challenge_pred = np.asarray(predict_texts(challenge_texts))
    unaccented_metric = metric_row(test_robust["label"], unaccented_pred)
    sensitivity_metric = metric_row(sensitivity_frame["label"], sensitivity_pred)
    pair_detail = evaluate_pairs(challenge_pred)
    row = {
        "seed": int(seed), "clean_test_macro_f1": float(json.loads(
            (checkpoint / "phobert_results.json").read_text(encoding="utf-8"))["test_macro_f1"]),
        "unaccented_test_n": int(len(test_robust)), "unaccented_test_overlap_n": int(overlap_mask_robust.sum()),
        "unaccented_test_macro_f1": unaccented_metric["macro_f1"],
        "unaccented_test_accuracy": unaccented_metric["accuracy"],
        "unaccented_test_complaint_precision": unaccented_metric["complaint_precision"],
        "unaccented_test_complaint_recall": unaccented_metric["complaint_recall"],
        "sensitivity_n": int(len(sensitivity_frame)),
        "sensitivity_macro_f1": sensitivity_metric["macro_f1"],
        "sensitivity_accuracy": sensitivity_metric["accuracy"],
        "challenge_pairs_correct": int(sum(item["both_correct"] for item in pair_detail)),
        "challenge_pairs_total": int(len(pair_detail)),
    }
    robust_rows.append(row)
    for item in pair_detail:
        pair_rows.append({"seed": int(seed), **item})
    pd.DataFrame({"test_row": test_robust.index, "label": test_robust["label"].to_numpy(),
                  "domain": test_robust["domain"].to_numpy(),
                  "prediction_unaccented": unaccented_pred}).to_csv(
        robust_out / f"unaccented_test_predictions_seed_{seed}.csv", index=False)
    del restored_model, restored_tokenizer
    gc.collect()
    torch.cuda.empty_cache()

robust_table = pd.DataFrame(robust_rows)
robust_table.to_csv(robust_out / "robustness_seed_metrics.csv", index=False)
pd.DataFrame(pair_rows).to_csv(robust_out / "challenge_predictions_by_seed.csv", index=False, encoding="utf-8-sig")
summary_robust = {"seeds": list(SEEDS), "n_runs": len(robust_rows),
    "test_unaccented_rows": int(len(test_robust)), "accent_overlap_rows": int(overlap_mask_robust.sum()),
    "sensitivity_rows": int(len(sensitivity_frame)),
    "metrics": {column: {"mean": float(robust_table[column].mean()),
                          "sd_sample": float(robust_table[column].std(ddof=1))}
                for column in ("clean_test_macro_f1", "unaccented_test_macro_f1",
                               "unaccented_test_accuracy", "sensitivity_macro_f1",
                               "challenge_pairs_correct")},
    "interpretation": "post-hoc exploratory evaluation on the fixed test split and synthetic diagnostic pairs; not independent out-of-domain confirmation"}
(robust_out / "robustness_summary.json").write_text(json.dumps(summary_robust, indent=2), encoding="utf-8")
display(robust_table.round(4))
display(pd.DataFrame([summary_robust]))


## Demo hệ thống PhoBERT

Sửa `review` ở cell kế tiếp để thử đánh giá khác. Checkpoint seed 42 được khôi phục một lần rồi dùng lại cho các lần dự đoán tiếp theo. Demo báo độ dài token và cảnh báo nếu cắt ở giới hạn mô hình. Điểm softmax chưa được hiệu chỉnh xác suất. Gọi `release_demo_checkpoint()` khi xong để giải phóng GPU.


In [ ]:
review = "Áo đẹp nhưng giao hàng quá chậm."
demo_reviews = [
    ("Review cần kiểm tra", review),
    ("Review tích cực", "Sản phẩm đẹp, chất lượng tốt, tôi rất hài lòng."),
]

for invalid_review in ("", "   ", None):
    try:
        predict_phobert_review(invalid_review)
    except (TypeError, ValueError):
        pass
    else:
        raise AssertionError("Demo phải từ chối dữ liệu không phải chuỗi và nội dung rỗng.")
print("Kiểm tra đầu vào rỗng/không phải chuỗi: đạt.")

for name, review_text in demo_reviews:
    result = predict_phobert_review(review_text)
    print(f"\n{name}")
    print("Đánh giá:", review_text)
    print(f"PhoBERT seed {result['seed']}: {result['prediction']}")
    print(
        f"Token đầu vào: {result['input_token_count']}; "
        f"được dùng: {result['used_token_count']}/{result['max_length']}; "
        f"bị cắt: {'có' if result['truncated'] else 'không'}."
    )
    if result["truncated"]:
        print("Cảnh báo: mô hình chỉ phân loại phần đầu trong giới hạn token.")
    print(f"Điểm softmax (chưa hiệu chỉnh): {result['softmax_score']:.4f}")
    print("Điểm theo nhãn:", json.dumps(result["probabilities"], ensure_ascii=False))

long_review = "Sản phẩm đẹp và giao hàng nhanh. " * 100
long_result = predict_phobert_review(long_review)
if not long_result["truncated"]:
    raise AssertionError("Kiểm tra review dài cần nhận được cờ báo cắt token.")
print(
    f"\nKiểm tra review dài: đã báo cắt "
    f"({long_result['input_token_count']} token đầu vào, "
    f"dùng {long_result['used_token_count']}/{long_result['max_length']})."
)
